In [1]:
import copy
import csv
import ast
from docplex.mp.model import Model
import random
import time
import heapq
import itertools
import math

In [2]:
K = 5 # smartagrisusy

In [3]:
counter = itertools.count()
join_tree = []
disjoint_tree = []
best_tree = []

# type 1: join, type 2: disjoint, type 3: best (solo bound)
def push(node, type=1):
    depth = node.depth if hasattr(node, 'depth') else 0
    priority = -depth  # più profondo = priorità più alta
    if type==1:
        heapq.heappush(join_tree, (priority, next(counter), node))
    elif type == 2:
        heapq.heappush(disjoint_tree, (-priority, next(counter), node))
    else:
        heapq.heappush(best_tree, (node.bound, next(counter), node))

def pop():
    if join_tree:
        _, _, node = heapq.heappop(join_tree)
    elif disjoint_tree:
        _, _, node = heapq.heappop(disjoint_tree)
    else:
        _, _, node = heapq.heappop(best_tree)
    return node

def tree_empty():
    return not join_tree and not best_tree and not disjoint_tree

def tree_size():
    return len(join_tree) + len(best_tree) + len(disjoint_tree)

In [4]:
def load_pksc_instance_from_csv(filepath, K, P):
    item_to_idx = {}
    idx_to_item = {}
    order_to_idx = {}
    idx_to_order = {}

    S = []
    I_s = {}

    with open(filepath, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)

        for row in reader:
            order_id = row['ORDER']
            product_list = ast.literal_eval(row['PRODUCT_LIST'])

            if order_id not in order_to_idx:
                s_idx = len(order_to_idx)
                order_to_idx[order_id] = s_idx
                idx_to_order[s_idx] = order_id
                S.append(s_idx)
                I_s[s_idx] = []
            else:
                s_idx = order_to_idx[order_id]

            for product in product_list:
                if product not in item_to_idx:
                    i_idx = len(item_to_idx)
                    item_to_idx[product] = i_idx
                    idx_to_item[i_idx] = product

                i_idx = item_to_idx[product]
                I_s[s_idx].append(i_idx)

    I = list(idx_to_item.keys())
    S_i = {i: [] for i in I}

    for s_idx, i_indices in I_s.items():
        for i_idx in i_indices:
            S_i[i_idx].append(s_idx)

    return {
        'K': K,
        'P': P,
        'I': I,
        'S': S,
        'I_s': {s: set(items) for s, items in I_s.items()},
        'S_i': {i: set(samples) for i, samples in S_i.items()},
        'idx_to_item': idx_to_item,
        'idx_to_order': idx_to_order
    }

In [5]:
def compute_h_mappings(H, I, S, I_s):
    H_i = {i: [] for i in I}

    for h_index, h_set in enumerate(H):
        for i in h_set:
            if i in H_i:
                H_i[i].append(h_index)

    H_s = {s: [] for s in S}
    for s in S:
        h_indices = set()
        for i in I_s[s]:
            h_indices.update(H_i[i])
        H_s[s] = list(h_indices)

    return H_i, H_s

In [6]:
def generate_initial_h(I):
    return [[i] for i in I]

In [7]:
def generate_initial_h_smart(S, I, k, I_s):
    added_patterns = set()

    def add_pattern(pattern):
        p_set = frozenset(pattern)
        if p_set not in added_patterns:
            added_patterns.add(p_set)

    for i_item in I:
        add_pattern({i_item})

    for s in S:
        s_list = sorted(list(I_s[s]))

        for chunk_start in range(0, len(s_list), k):
            chunk = s_list[chunk_start:chunk_start + k]
            if 1 <= len(chunk) <= k:
                add_pattern(chunk)

        if k >= 2:
            if len(s_list) <= 50:
                for combo in combinations(s_list, min(k, len(s_list))):
                    add_pattern(combo)
            else:
                random.seed(42)
                sampled = random.sample(s_list, min(30, len(s_list)))
                for combo in combinations(sampled, min(k, len(sampled))):
                    add_pattern(combo)

    return list(added_patterns)

In [8]:
def is_integer(x, tol=1e-6):
    for x_h in x:
            if tol < x[x_h].solution_value < 1 - tol:
                return False
    return True

In [9]:
def setup_rmp(model, instance, H, H_i, H_s):

    x = {j: model.continuous_var(name=f'x_{j}') for j in range(len(H))}
    P = model.continuous_var(name='P')

    constraints_alpha = {}
    for i in instance['I']:
        constraints_alpha[i] = model.add_constraint(model.sum(x[j] for j in H_i[i]) == 1, f'cover_item_{i}')

    constraints_beta = {}
    for s in instance['S']:
        constraints_beta[s] = model.add_constraint(model.sum(x[h] for h in H_s[s]) <= P, f'price_order_{s}')

    model.minimize(P)

    return x, P, constraints_alpha, constraints_beta

In [10]:
def add_new_pattern_to_rmp(model, H, new_h, x, constraints_alpha, constraints_beta, instance):
    new_h_index = len(H)
    H.append(new_h)
    x[new_h_index] = model.continuous_var(name=f'x_{new_h_index}')

    covered_samples = set()
    for i in new_h:
        constraints_alpha[i].left_expr.add_term(x[new_h_index], 1)
        covered_samples.update(instance['S_i'][i])

    for s in covered_samples:
        constraints_beta[s].left_expr.add_term(x[new_h_index], 1)

In [11]:
def setup_pricing_problem(model, instance, constraints=None):
    model.set_time_limit(60)
    model.context.cplex_parameters.threads = 1
    model.context.cplex_parameters.mip.tolerances.mipgap = 0.01
    model.context.cplex_parameters.emphasis.mip = 1

    z = {i: model.binary_var(name=f"z_{i}") for i in instance['I']}

    relevant_samples = set()
    for i in instance['I']:
        relevant_samples.update(instance['S_i'][i])

    w = {s: model.binary_var(name=f"w_{s}") for s in relevant_samples}

    model.add_constraint(model.sum(z[i] for i in instance['I']) <= instance['K'], ctname="size_limit")

    for i in instance['I']:
        for s in instance['S_i'][i]:
            model.add_constraint(w[s] >= z[i], ctname=f"link_{s}_{i}")

    if constraints:
        for c in constraints:
            if c[0] == 'join':
                i, j = c[1]
                model.add_constraint(z[i] == z[j], ctname=f"join_{i}_{j}")
            elif c[0] == 'disjoint':
                i, j = c[1]
                model.add_constraint(z[i] + z[j] <= 1, ctname=f"disjoint_{i}_{j}")

    return z, w

In [12]:
def update_pricing_problem(model, instance, duals_alpha, duals_beta, z, w):
    model.set_objective('max', model.sum(duals_alpha[i] * z[i] for i in instance['I']) + model.sum(duals_beta[s] * w[s] for s in w))

In [13]:
def solve_rmp(model, constraints_alpha, constraints_beta):
    model.solve()

    if model.solution is None:
        return None, None, None

    duals_alpha = {i: constraints_alpha[i].dual_value for i in instance['I']}
    duals_beta  = {s: constraints_beta[s].dual_value  for s in instance['S']}

    return model.solution.get_objective_value(), duals_alpha, duals_beta

In [14]:
def solve_pricing_problem(model, z):
    model.solve()
    if model.solution is None:
        return 0, None
    if model.solution.get_objective_value() <= 1e-6:
        return 0, None
    selected_items = {i for i, var in z.items() if var.solution_value > 0.5}
    return model.solution.get_objective_value(), selected_items

In [15]:
def compute_join_groups(previous_constraints):
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent.setdefault(x, x)
        parent.setdefault(y, y)
        parent[find(x)] = find(y)

    for c in previous_constraints:
        if c[0] == 'join':
            union(c[1][0], c[1][1])

    groups = defaultdict(set)
    for x in parent:
        groups[find(x)].add(x)

    return groups

In [16]:
def set_models_parameters(rmp_model, pricing_model):
    rmp_model.context.cplex_parameters.lpmethod = 2
    rmp_model.context.cplex_parameters.preprocessing.presolve = 1
    rmp_model.context.solver.log_output = False
    pricing_model.context.solver.log_output = False
    pricing_model.set_time_limit(60)
    pricing_model.context.cplex_parameters.threads = 1
    pricing_model.context.cplex_parameters.mip.tolerances.mipgap = 0.01
    pricing_model.context.cplex_parameters.emphasis.mip = 1

In [17]:
def column_generation(instance, H, iterations=1500, constraints = None):
    H_i, H_s = compute_h_mappings(H, instance['I'], instance['S'], instance['I_s'])

    model_rmp = Model(name="RMP")
    x, P, constraints_alpha, constraints_beta = setup_rmp(model_rmp, instance, H, H_i, H_s)

    model_pricing = Model(name="Pricing")
    z, w = setup_pricing_problem(model_pricing, instance, constraints)

    set_models_parameters(model_rmp, model_pricing)

    rmp_time = 0
    updating_time = 0
    pricing_time = 0
    adding_time = 0
    rmp_obj = 0

    begin_column = time.time()
    iteration = 0
    for iteration in range(iterations):

        start = time.time()
        rmp_obj, duals_alpha, duals_beta = solve_rmp(model_rmp, constraints_alpha, constraints_beta)
        rmp_time += time.time() - start

        if rmp_obj is None:
            return None, None, None, None, 0, float('inf'), H, 0, {}

        start = time.time()
        update_pricing_problem(model_pricing, instance, duals_alpha, duals_beta, z, w)
        updating_time += time.time() - start
        start = time.time()
        pricing_obj, new_h = solve_pricing_problem(model_pricing, z)
        pricing_time += time.time() - start

        if pricing_obj is None or pricing_obj <= 1e-6:
            #print(f"Optimal solution found at iteration {iteration} with objective value {rmp_obj:.4f}")
            break

        start = time.time()
        add_new_pattern_to_rmp(model_rmp, H, new_h, x, constraints_alpha, constraints_beta, instance)
        adding_time += time.time() - start

        #print(f"Iteration {iteration} | Reduced cost {pricing_obj:.4f}, RMP objective value: {rmp_obj:.4f}")
    end_column = time.time() - begin_column
    return rmp_time, updating_time, pricing_time, adding_time, end_column, rmp_obj, H, iteration, x

In [18]:
def print_result(results):
    print("Column generation results:")
    print(f"Total RMP solving time: {results[0]:.2f} seconds")
    print(f"Total pricing problem updating time: {results[1]:.2f} seconds")
    print(f"Total pricing problem solving time: {results[2]:.2f} seconds")
    print(f"Total time adding new patterns to RMP: {results[3]:.2f} seconds")
    print(f"Total time of column generation: {results[4]:.2f} seconds")
    print(f"Final RMP objective value: {results[5]:.4f}")
    print(f"Number of patterns in final RMP: {len(results[6])}")
    print(f"Number of iterations: {results[7]}")

In [19]:
def select_pair_to_branch_on(x, H, strategy='most_fractional', previous_constraints=None):
    # Calcola coppie già vincolate
    already_constrained = set()
    if previous_constraints:
        groups = compute_join_groups(previous_constraints)
        disjoint_pairs = {frozenset(c[1]) for c in previous_constraints if c[0] == 'disjoint'}

        for members in groups.values():
            for a, b in combinations(members, 2):
                already_constrained.add(frozenset((a, b)))
        already_constrained.update(disjoint_pairs)

    scores = defaultdict(float)
    for h_idx, h in enumerate(H):
        x_val = x[h_idx].solution_value
        if x_val <= 1e-6:
            continue
        items_in_h = list(h)
        for a in range(len(items_in_h)):
            for b in range(a + 1, len(items_in_h)):
                i, j = items_in_h[a], items_in_h[b]
                if frozenset((i, j)) not in already_constrained:  # ← skip coppie già vincolate
                    scores[(i, j)] += x_val

    if not scores:
        return None

    if strategy == 'most_fractional':
        return min(scores, key=lambda pair: abs(scores[pair] - 0.5))
    elif strategy == 'max_score':
        return max(scores, key=scores.get)
    elif strategy == 'min_score':
        return min(scores, key=scores.get)
    return None

In [20]:
from collections import defaultdict
from itertools import combinations

def get_relations(all_constraints):
    parent = {}

    # Union-Find per i gruppi JOIN
    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent.setdefault(x, x)
        parent.setdefault(y, y)
        parent[find(x)] = find(y)

    conflict_map = defaultdict(set)

    for c_type, pair in all_constraints:
        u, v = tuple(pair)
        if c_type == 'join':
            union(u, v)
        elif c_type == 'disjoint':
            conflict_map[u].add(v)
            conflict_map[v].add(u)

    groups = defaultdict(set)
    for x in parent:
        groups[find(x)].add(x)

    def get_group(item):
        for rep, members in groups.items():
            if item in members:
                return members
        return {item}

    return get_group, conflict_map

def get_join_groups(constraints):
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent.setdefault(x, x)
        parent.setdefault(y, y)
        parent[find(x)] = find(y)

    for c_type, (u, v) in constraints:
        if c_type == 'join':
            union(u, v)

    groups = defaultdict(set)
    for x in list(parent.keys()):
        groups[find(x)].add(x)

    def get_group(item):
        if item not in parent:
            return frozenset({item})
        return frozenset(groups[find(item)])

    return get_group


def generate_left_h(H, a, b, K, previous_constraints):
    get_group = get_join_groups(previous_constraints)

    group_a = get_group(a)
    group_b = get_group(b)
    merged = group_a | group_b

    if len(merged) > K:
        return None

    temp_constraints = previous_constraints + [('join', (a, b))]
    get_group_new = get_join_groups(temp_constraints)

    H_new = set()

    for h in H:
        h_set = frozenset(h)

        touched_groups = {}
        for item in h_set:
            g = get_group_new(item)
            rep = min(g)
            touched_groups[rep] = g

        new_h = set()
        for rep, group in touched_groups.items():
            new_h |= group

        if len(new_h) <= K:
            H_new.add(frozenset(new_h))
        else:
            for rep, group in touched_groups.items():
                if len(group) <= K:
                    H_new.add(frozenset(group))

    H_new.add(frozenset(merged))

    result = [list(h) for h in H_new]
    return result


def generate_right_h(H, a, b, previous_constraints):
    temp_constraints = previous_constraints + [('disjoint', (a, b))]
    get_group = get_join_groups(previous_constraints)

    group_a = get_group(a)
    group_b = get_group(b)

    if group_a & group_b:
        return None

    H_new = set()

    for h in H:
        h_set = frozenset(h)

        has_a_group = bool(h_set & group_a)
        has_b_group = bool(h_set & group_b)

        if not (has_a_group and has_b_group):
            H_new.add(h_set)
        else:
            h_without_b = h_set - group_b
            if h_without_b:
                H_new.add(frozenset(h_without_b))

            h_without_a = h_set - group_a
            if h_without_a:
                H_new.add(frozenset(h_without_a))

    result = [list(h) for h in H_new]
    return result

In [21]:
def simple_prune_H(H, x):
    pruned_H = []
    for h_idx, h in enumerate(H):
        if x[h_idx].solution_value > 1e-6:
            pruned_H.append(h)
    return pruned_H

In [22]:
class Node:
    def __init__(self, bound, H, constraint, depth):
        self.bound = bound      # bound del PADRE (ottimistico, per ordinare la coda)
        self.H = H
        self.constraint = constraint
        self.depth = depth
        # results rimosso: si calcola lazy

class SolutionNode(Node):
    def __init__(self, bound, H, constraint, results, depth):
        super().__init__(bound, H, constraint, depth=depth)
        self.results = results


In [23]:
def verify_integer_solution(solution_node, instance):
    print("\n" + "="*60)
    print("Check Integer solution")
    print("="*60)

    results = solution_node.results
    x = results[8]
    H = results[6]
    K = instance['K']

    active_patterns = []
    for h_idx, h in enumerate(H):
        val = x[h_idx].solution_value
        if val > 0.5:
            active_patterns.append((h_idx, list(h), val))

    print("\n[CHECK 1] Pattern <= K")
    violations_k = []
    for h_idx, h, val in active_patterns:
        if len(h) > K:
            violations_k.append((h_idx, h, len(h)))
            print(f"  ✗ Pattern {h_idx}: size={len(h)} > K={K} | items={h}")


    if not violations_k:
        print(f"  → All {len(active_patterns)} patterns respect K={K}")

    print("\n[CHECK 2] Cover item (exact one)")
    item_coverage = defaultdict(list)
    for h_idx, h, val in active_patterns:
        for item in h:
            item_coverage[item].append(h_idx)

    violations_coverage = []
    uncovered = []

    for item in instance['I']:
        count = len(item_coverage[item])
        if count == 0:
            uncovered.append(item)
        elif count > 1:
            violations_coverage.append((item, item_coverage[item]))

    if not violations_coverage and not uncovered:
        print(f"  → All {len(instance['I'])} items covered exact once ✓")
    else:
        if uncovered:
            print(f"  ✗ Item NOT covered ({len(uncovered)}): {uncovered}")
        if violations_coverage:
            print(f"  ✗ Item covered more than once ({len(violations_coverage)}):")
            for item, patterns in violations_coverage:
                print(f"    item {item} covered by pattern {patterns}")

    print("\n[CHECK 3] Check obj (P = max pattern price)")
    H_i, H_s = compute_h_mappings(H, instance['I'], instance['S'], instance['I_s'])

    order_prices = {}
    for s in instance['S']:
        price = sum(
            x[h_idx].solution_value
            for h_idx in H_s[s]
            if x[h_idx].solution_value > 0.5
        )
        order_prices[s] = price

    computed_P = max(order_prices.values()) if order_prices else 0
    reported_P = results[5]

    print(f"  P from solver : {reported_P:.4f}")
    print(f"  P recomputed  : {computed_P:.4f}")
    if abs(computed_P - reported_P) < 1e-4:
        print(f"  → Obj checked ✓")
    else:
        print(f"  ✗ Error in obj: {abs(computed_P - reported_P):.6f}")

    print("\n" + "="*60)
    ok = not violations_k and not violations_coverage and not uncovered and abs(computed_P - reported_P) < 1e-4
    if ok:
        print(f"✓ Valid solution | P = {reported_P:.4f} | pattern = {len(active_patterns)}")
    else:
        print(f"✗ Not Valid solution")
    print("="*60 + "\n")

    return ok

In [24]:
def start_branch_and_price(instance):
    H = generate_initial_h_smart(instance['S'], instance['I'], instance['K'], instance['I_s'])
    previous_constraints = []

    root_results = column_generation(instance, H, constraints=previous_constraints)
    H = simple_prune_H(H, root_results[8])

    root = Node(bound=root_results[5], H=H, constraint=previous_constraints, depth=0)
    root.results = root_results
    push(root, type=3)
    print_result(root_results)

    best_solution = SolutionNode(bound=float('inf'), H=[], constraint=[], results=None, depth=0)

    best_lower_bound = math.ceil(root_results[5])
    start_tree_exploration = time.time()

    while not tree_empty():
        node = pop()

        print(f"Processing node with bound {node.bound:.7f}, depth={node.depth}, "
              f"patterns={len(node.H)}")

        # Pota subito con il bound ottimistico (ereditato dal padre)
        if math.ceil(node.bound) >= best_solution.bound:
            print("Pruning by bound before CG.")
            continue

        # Solo ora risolvi il CG
        results = column_generation(instance, node.H, constraints=node.constraint)
        node.bound = results[5]  # aggiorna con il bound reale

        print(f"Processed node at depth {node.depth} | CG bound: {node.bound:.4f} | patterns: {len(node.H)} | CG time: {results[4]:.2f} seconds")

        if results[5] == float('inf'):
            print("Unfeasible after CG")
            continue

        if math.ceil(node.bound - 1e-6) >= best_solution.bound:
            print("Pruning by bound after CG.")
            continue

        if is_integer(results[8]):
            print(f"Integer solution found with objective value {results[5]:.4f} found after {time.time() - start_tree_exploration:.4f} seconds.")
            best_solution = SolutionNode(
                bound=results[5], H=node.H,
                constraint=node.constraint.copy(), results=results, depth=node.depth
            )
            if best_solution.bound == best_lower_bound:
                print("Optimal integer solution found, terminating.")
                break
            continue

        pair = select_pair_to_branch_on(
            results[8], results[6], previous_constraints=node.constraint
        )

        if pair is None:
            print("No more pairs to branch on.")
            continue


        left_constraint  = node.constraint.copy() + [('join',     pair)]
        right_constraint = node.constraint.copy() + [('disjoint', pair)]

        current_depth = node.depth + 1

        H_left = generate_left_h(copy.deepcopy(node.H), pair[0], pair[1], instance['K'], left_constraint)
        if H_left:
            left_node = Node(bound=node.bound, H=H_left, constraint=left_constraint, depth=current_depth)
            push(left_node, type=1)

        H_right = generate_right_h(node.H.copy(), pair[0], pair[1], right_constraint)
        if H_right:
            right_node = Node(bound=node.bound, H=H_right, constraint=right_constraint, depth=current_depth)
            if node.bound == best_solution.bound - 1:
                push(right_node, type=2)
            else:
                push(right_node, type=3)


    print(f"Branch-and-Price completed in {(time.time() - start_tree_exploration)} seconds.")


    if best_solution.results is not None:
        verify_integer_solution(best_solution, instance)
    else:
        print("No integer solution found.")



In [25]:
instance = load_pksc_instance_from_csv('filtered_dataset.csv', K=K, P=10)

start_branch_and_price(instance)

Column generation results:
Total RMP solving time: 7.20 seconds
Total pricing problem updating time: 0.47 seconds
Total pricing problem solving time: 33.70 seconds
Total time adding new patterns to RMP: 1.17 seconds
Total time of column generation: 42.55 seconds
Final RMP objective value: 3.1866
Number of patterns in final RMP: 20980
Number of iterations: 350
Processing node with bound 3.1866349, depth=0, patterns=1072
Processed node at depth 0 | CG bound: 3.1866 | patterns: 1087 | CG time: 0.67 seconds
Processing node with bound 3.1866349, depth=1, patterns=1089
Processed node at depth 1 | CG bound: 3.1866 | patterns: 1097 | CG time: 0.24 seconds
Processing node with bound 3.1866349, depth=2, patterns=1100
Processed node at depth 2 | CG bound: 3.1866 | patterns: 1101 | CG time: 0.06 seconds
Processing node with bound 3.1866349, depth=3, patterns=1105
Processed node at depth 3 | CG bound: 3.1866 | patterns: 1113 | CG time: 0.14 seconds
Processing node with bound 3.1866349, depth=4, pat